# 02. Feature Engineering

목적: **주차별 그룹 목표** 학습 데이터 생성

개선 사항:
1. 불필요한 컬럼 제거 (22개 → 9개)
2. 그룹 크기 랜덤화 (2-6명, fitness_level 기준 정렬)
3. 일별 → **주차별** 집계
4. `group_weekly_active_rate` 피처 추가

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# 원본 데이터 로드
df = pd.read_csv('../data/raw/health_fitness_dataset.csv')

print(f"원본 데이터 크기: {df.shape}")
df.head()

: 

In [ ]:
# 1. 불필요한 컬럼 제거
columns_to_keep = [
    'participant_id', 'date', 'age', 'bmi', 'resting_heart_rate',
    'daily_steps', 'calories_burned', 'duration_minutes', 'fitness_level'
]

df = df[columns_to_keep].copy()
print(f"컬럼 제거 후: {df.shape}")
print(f"유지된 컬럼: {list(df.columns)}")

In [ ]:
# 2. 날짜 변환 및 계절 계산
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['week'] = df['date'].dt.isocalendar().week
df['month'] = df['date'].dt.month

def get_season(month):
    """월 → 계절 (1~4)"""
    if month in [3, 4, 5]:
        return 1  # 봄
    elif month in [6, 7, 8]:
        return 2  # 여름
    elif month in [9, 10, 11]:
        return 3  # 가을
    else:
        return 4  # 겨울

df['season'] = df['month'].apply(get_season)

# 거리 계산 (1000보 ≈ 0.8km)
df['distance_km'] = df['daily_steps'] * 0.8 / 1000

print(f"날짜 변환 완료")
df[['date', 'year', 'week', 'month', 'season']].head()

In [ ]:
# 3. 참가자별 평균 fitness_level 계산
participant_fitness = df.groupby('participant_id').agg({
    'fitness_level': 'mean'
}).reset_index()
participant_fitness.columns = ['participant_id', 'avg_fitness_level']

print(f"참가자 수: {len(participant_fitness)}")
print(f"Fitness level 범위: {participant_fitness['avg_fitness_level'].min():.2f} ~ {participant_fitness['avg_fitness_level'].max():.2f}")
participant_fitness.head()

In [ ]:
# 4. 그룹 생성 (fitness_level 기준, 2-6명 랜덤)
def create_groups_by_fitness_level(participants_df, min_size=2, max_size=6, seed=42):
    """
    참가자들을 fitness_level 기준으로 정렬 후 랜덤 크기(2-6명)의 그룹으로 묶기
    """
    np.random.seed(seed)
    
    # fitness_level 기준 정렬
    participants_sorted = participants_df.sort_values('avg_fitness_level')
    participant_ids = participants_sorted['participant_id'].values
    
    group_mapping = {}
    group_id = 1
    i = 0
    
    while i < len(participant_ids):
        remaining = len(participant_ids) - i
        
        # 남은 인원이 min_size 미만이면 이전 그룹에 병합
        if remaining < min_size and i > 0:
            group_id -= 1
        
        # 그룹 크기 랜덤 결정
        if remaining <= max_size:
            group_size = remaining
        else:
            max_possible = min(max_size, remaining - min_size)
            group_size = np.random.randint(min_size, max_possible + 1)
        
        # 그룹 할당
        for j in range(i, min(i + group_size, len(participant_ids))):
            group_mapping[participant_ids[j]] = group_id
        
        i += group_size
        group_id += 1
    
    return group_mapping

# 그룹 생성
group_mapping = create_groups_by_fitness_level(participant_fitness)
df['group_id'] = df['participant_id'].map(group_mapping)

# 그룹 크기 통계
group_sizes = df.groupby('group_id')['participant_id'].nunique()
print(f"생성된 그룹 수: {len(group_sizes)}")
print(f"\n그룹 크기 분포:")
print(group_sizes.value_counts().sort_index())

In [ ]:
# 5. 주차별 그룹 집계
weekly_group_stats = df.groupby(['group_id', 'year', 'week']).agg({
    'participant_id': 'nunique',  # 그룹 인원수
    'daily_steps': 'mean',
    'calories_burned': 'mean',
    'duration_minutes': 'mean',
    'distance_km': 'mean',
    'age': 'mean',
    'bmi': 'mean',
    'resting_heart_rate': 'mean',
    'season': 'first',
    'date': 'count'  # 활동 일수
}).reset_index()

weekly_group_stats.columns = [
    'group_id', 'year', 'week', 'member_count',
    'group_avg_steps_14', 'group_avg_kcal_14', 'group_avg_duration_14', 'group_avg_distance_14',
    'avg_age', 'avg_bmi', 'avg_resting_hr', 'season', 'active_days'
]

print(f"주차별 그룹 데이터: {weekly_group_stats.shape}")
weekly_group_stats.head()

In [ ]:
# 6. group_weekly_active_rate 계산
weekly_group_stats['group_weekly_active_rate'] = (weekly_group_stats['active_days'] / 7).round(2)

# 7. 이전 주 달성률 계산 (임시: 노이즈 추가)
np.random.seed(42)
weekly_group_stats['last_week_achievement_rate'] = np.random.uniform(0.5, 1.2, len(weekly_group_stats)).round(2)

# 8. user_min 값 계산 (그룹 평균의 60%)
weekly_group_stats['user_min_steps'] = (weekly_group_stats['group_avg_steps_14'] * 0.6).round()
weekly_group_stats['user_min_kcal'] = (weekly_group_stats['group_avg_kcal_14'] * 0.6).round(1)
weekly_group_stats['user_min_duration'] = (weekly_group_stats['group_avg_duration_14'] * 0.6).round()
weekly_group_stats['user_min_distance'] = (weekly_group_stats['group_avg_distance_14'] * 0.6).round(2)

print("피처 추가 완료")
weekly_group_stats[['group_id', 'week', 'group_weekly_active_rate', 'last_week_achievement_rate']].head()

In [ ]:
# ============================================================
# Step 1: 개인별 주차 데이터 계산
# ============================================================
print("\n📊 Step 1: 개인별 주차 데이터 계산 중...")

individual_weekly = df.groupby(['participant_id', 'group_id', 'year', 'week']).agg({
    'daily_steps': 'mean',
    'calories_burned': 'mean',
    'duration_minutes': 'mean',
    'distance_km': 'mean',
    'date': 'count'  # 활동일 수
}).reset_index()

# 컬럼명 변경
individual_weekly.columns = [
    'participant_id', 'group_id', 'year', 'week',
    'ind_avg_steps', 'ind_avg_kcal', 'ind_avg_duration', 'ind_avg_distance',
    'ind_active_days'
]

# 개인별 활동일 비율
individual_weekly['ind_active_days_rate'] = (individual_weekly['ind_active_days'] / 7).round(2)

print(f"   ✅ 개인별 주차 데이터: {len(individual_weekly)} rows")
print(f"   컬럼: {list(individual_weekly.columns)}")
individual_weekly.head()

In [ ]:
# ============================================================
# Step 2: 개인별 성장률 계산 (최근 3~4주)
# ============================================================
print("\n📈 Step 2: 개인별 성장률 계산 중...")

# 정렬 (시계열 순서 보장)
individual_weekly = individual_weekly.sort_values(['participant_id', 'year', 'week'])

# 각 개인의 지난 주 대비 성장률
for metric in ['steps', 'kcal', 'duration', 'distance']:
    # pct_change()는 증가율(-1~1)을 반환하므로 +1하면 비율(0~2)
    individual_weekly[f'ind_growth_rate_{metric}'] = (
        individual_weekly.groupby('participant_id')[f'ind_avg_{metric}']
        .pct_change() + 1
    )
    
    # 결측값 처리 (첫 주는 성장률 계산 불가 → 1.0으로 초기화)
    individual_weekly[f'ind_growth_rate_{metric}'].fillna(1.0, inplace=True)
    
    # 이상치 제거 (0.3 ~ 2.0 범위로 제한)
    individual_weekly[f'ind_growth_rate_{metric}'] = individual_weekly[f'ind_growth_rate_{metric}'].clip(0.3, 2.0)

print(f"   ✅ 성장률 컬럼 추가: ind_growth_rate_steps, ind_growth_rate_kcal, etc.")
print(f"   Steps 성장률 통계:")
print(individual_weekly['ind_growth_rate_steps'].describe())
individual_weekly[['participant_id', 'year', 'week', 'ind_avg_steps', 'ind_growth_rate_steps']].head(10)

In [ ]:
# ============================================================
# Step 3: 개인별 달성률 계산
# ============================================================
print("\n🎯 Step 3: 개인별 달성률 계산 중...")

# 주의: 실제 시스템에서는 실제 개인 목표 사용
# 학습 데이터에서는 그룹 평균을 개인 목표로 가정

for metric in ['steps', 'kcal', 'duration', 'distance']:
    # 그룹 평균 매핑
    individual_weekly = individual_weekly.merge(
        weekly_group_stats[['group_id', 'year', 'week', f'group_avg_{metric}_14']],
        on=['group_id', 'year', 'week'],
        how='left',
        suffixes=('', '_group')
    )
    
    # 개인 달성률 = 개인 실제 / 그룹 평균 (목표로 간주)
    individual_weekly[f'ind_achievement_{metric}'] = (
        individual_weekly[f'ind_avg_{metric}'] / 
        individual_weekly[f'group_avg_{metric}_14']
    ).fillna(1.0)
    
    # 이상치 제거 (0.1 ~ 3.0 범위)
    individual_weekly[f'ind_achievement_{metric}'] = individual_weekly[f'ind_achievement_{metric}'].clip(0.1, 3.0)
    
    # 중복 컬럼 제거 (group_avg_*_14는 이미 weekly_group_stats에 있음)
    individual_weekly = individual_weekly.drop(columns=[f'group_avg_{metric}_14'], errors='ignore')

print(f"   ✅ 달성률 컬럼 추가: ind_achievement_steps, ind_achievement_kcal, etc.")
print(f"   Steps 달성률 통계:")
print(individual_weekly['ind_achievement_steps'].describe())
individual_weekly[['participant_id', 'year', 'week', 'ind_avg_steps', 'ind_achievement_steps']].head(10)

In [ ]:
# ============================================================
# Step 4: 개인별 달성률 추세 계산 (최근 3~4주 기울기)
# ============================================================
print("\n📉 Step 4: 개인별 달성률 추세 계산 중...")

def calculate_trend(series, window=4):
    """
    최근 window 주간의 선형 회귀 기울기 계산
    
    Args:
        series: 달성률 시계열 (pandas Series)
        window: 최근 몇 주를 볼 것인지 (기본 4주)
    
    Returns:
        float: 선형 회귀 기울기 (양수=개선, 음수=피로 누적)
    """
    if len(series) < 2:
        return 0.0
    
    # 최근 window개만 사용
    series = series.iloc[-window:]
    if len(series) < 2:
        return 0.0
    
    x = np.arange(len(series))
    y = series.values
    
    # 모두 같은 값이면 기울기 0
    if np.std(y) == 0:
        return 0.0
    
    # 선형 회귀 기울기 계산
    slope = np.polyfit(x, y, 1)[0]
    return slope

# 각 개인의 달성률 추세 계산 (steps 기준)
individual_weekly['ind_trend_achievement'] = (
    individual_weekly.groupby('participant_id')['ind_achievement_steps']
    .transform(lambda x: x.rolling(window=4, min_periods=2).apply(
        lambda s: calculate_trend(s, window=len(s))
    ))
).fillna(0.0)

# 이상치 제거 (-0.5 ~ 0.5 범위)
individual_weekly['ind_trend_achievement'] = individual_weekly['ind_trend_achievement'].clip(-0.5, 0.5)

print(f"   ✅ 추세 컬럼 추가: ind_trend_achievement")
print(f"   달성률 추세 통계:")
print(individual_weekly['ind_trend_achievement'].describe())
print(f"   해석:")
print(f"     - 양수: 개인들이 개선되고 있음 (달성률 상승)")
print(f"     - 음수: 개인들이 피로 누적 중 (달성률 하락) → 번아웃 신호")
individual_weekly[['participant_id', 'year', 'week', 'ind_achievement_steps', 'ind_trend_achievement']].tail(20)

In [ ]:
# ============================================================
# Step 5: 개인 추세 → 그룹 Feature로 집계
# ============================================================
print("\n🔄 Step 5: 개인 추세를 그룹 Feature로 집계 중...")

member_trends = individual_weekly.groupby(['group_id', 'year', 'week']).agg({
    # 성장률 통계 (평균, 분산)
    'ind_growth_rate_steps': ['mean', 'var'],
    'ind_growth_rate_kcal': ['mean', 'var'],
    'ind_growth_rate_duration': ['mean', 'var'],
    'ind_growth_rate_distance': ['mean', 'var'],
    
    # 달성률 통계 (평균)
    'ind_achievement_steps': 'mean',
    
    # 달성률 추세 (평균)
    'ind_trend_achievement': 'mean',
    
    # 활동률 통계 (평균)
    'ind_active_days_rate': 'mean'
}).reset_index()

# 컬럼명 정리 (MultiIndex 제거)
member_trends.columns = [
    'group_id', 'year', 'week',
    'avg_growth_rate_members_steps', 'var_growth_rate_members_steps',
    'avg_growth_rate_members_kcal', 'var_growth_rate_members_kcal',
    'avg_growth_rate_members_duration', 'var_growth_rate_members_duration',
    'avg_growth_rate_members_distance', 'var_growth_rate_members_distance',
    'avg_achievement_members',
    'trend_achievement_members',
    'avg_active_days_rate_members'
]

print(f"   ✅ 그룹별 개인 추세 집계: {len(member_trends)} rows")
print(f"   새로운 컬럼:")
for col in member_trends.columns[3:]:  # group_id, year, week 제외
    print(f"     - {col}")

# 그룹 통계와 병합
print("\n   병합 전 weekly_group_stats 크기:", weekly_group_stats.shape)
weekly_group_stats = weekly_group_stats.merge(
    member_trends,
    on=['group_id', 'year', 'week'],
    how='left'
)
print("   병합 후 weekly_group_stats 크기:", weekly_group_stats.shape)

# 결측값 처리 (첫 주 등)
print("\n   결측값 처리 중...")
weekly_group_stats['avg_growth_rate_members_steps'].fillna(1.0, inplace=True)
weekly_group_stats['var_growth_rate_members_steps'].fillna(0.01, inplace=True)
weekly_group_stats['avg_growth_rate_members_kcal'].fillna(1.0, inplace=True)
weekly_group_stats['var_growth_rate_members_kcal'].fillna(0.01, inplace=True)
weekly_group_stats['avg_growth_rate_members_duration'].fillna(1.0, inplace=True)
weekly_group_stats['var_growth_rate_members_duration'].fillna(0.01, inplace=True)
weekly_group_stats['avg_growth_rate_members_distance'].fillna(1.0, inplace=True)
weekly_group_stats['var_growth_rate_members_distance'].fillna(0.01, inplace=True)
weekly_group_stats['avg_achievement_members'].fillna(1.0, inplace=True)
weekly_group_stats['trend_achievement_members'].fillna(0.0, inplace=True)
weekly_group_stats['avg_active_days_rate_members'].fillna(0.5, inplace=True)

print("   ✅ 결측값 처리 완료")

# 통계 확인
print("\n   📊 개인 추세 Feature 통계:")
print(weekly_group_stats[['avg_growth_rate_members_steps', 'var_growth_rate_members_steps', 
                          'avg_achievement_members', 'trend_achievement_members']].describe())

weekly_group_stats.head()

In [ ]:
# ============================================================
# Step 6: 성장률 타겟 생성 (절대값 → 성장률)
# ============================================================
print("\n🎯 Step 6: 성장률 타겟 계산 중...")

# 주차별 데이터를 정렬 (이미 정렬되어 있지만 확인)
weekly_group_stats = weekly_group_stats.sort_values(['group_id', 'year', 'week']).reset_index(drop=True)

# ✅ 새로운 방식: 성장률 계산
for metric in ['steps', 'kcal', 'duration', 'distance']:
    # 다음 주 값을 먼저 구함
    weekly_group_stats[f'next_week_{metric}'] = weekly_group_stats.groupby('group_id')[f'group_avg_{metric}_14'].shift(-1)
    
    # 성장률 = 다음 주 평균 / 이번 주 평균
    weekly_group_stats[f'growth_rate_{metric}'] = (
        weekly_group_stats[f'next_week_{metric}'] /
        weekly_group_stats[f'group_avg_{metric}_14']
    )
    
    # 이상치 제거 (0.5 ~ 1.5 범위로 제한)
    # 0.5 = -50% (급격한 감소), 1.5 = +50% (급격한 증가)
    weekly_group_stats[f'growth_rate_{metric}'] = weekly_group_stats[f'growth_rate_{metric}'].clip(0.5, 1.5)

# 다음 주 데이터가 없는 행 제거 (마지막 주)
weekly_group_stats = weekly_group_stats.dropna(subset=['growth_rate_steps']).copy()

# 성장률 통계 출력
print("\n   📊 성장률 통계:")
for metric in ['steps', 'kcal', 'duration', 'distance']:
    mean_growth = weekly_group_stats[f'growth_rate_{metric}'].mean()
    median_growth = weekly_group_stats[f'growth_rate_{metric}'].median()
    std_growth = weekly_group_stats[f'growth_rate_{metric}'].std()
    print(f"   {metric.capitalize():12} - 평균: {mean_growth:.3f}, 중앙값: {median_growth:.3f}, 표준편차: {std_growth:.3f}")

print("\n   ✅ 성장률 타겟 생성 완료")
print(f"   총 데이터: {len(weekly_group_stats)} rows")
print(f"   해석: 1.0 = 유지, 1.1 = 10% 성장, 0.9 = 10% 감소")

weekly_group_stats[['group_id', 'year', 'week', 'group_avg_steps_14', 'growth_rate_steps']].head(20)

In [ ]:
# 11. 데이터 확인 및 분석

print("="*60)
print("📊 최종 데이터 확인")
print("="*60)

print("\n=== 최종 피처 (9개) ===")
feature_names = [
    'member_count', 'group_avg_steps_14', 'last_week_achievement_rate',
    'avg_age', 'avg_bmi', 'avg_resting_hr', 'season',
    'group_weekly_active_rate', 'user_min_steps'
]
for i, f in enumerate(feature_names, 1):
    print(f"{i}. {f}")

print("\n=== 타겟 변수 ===")

print("next_goal_steps (다음 주 실제 달성값 × 그룹 인원)")
print(weekly_group_stats.columns.tolist())

# training_steps가 아직 없으면 여기서 생성
if 'training_steps' not in locals():
    steps_cols = [
        'member_count', 'group_avg_steps_14', 'last_week_achievement_rate',
        'avg_age', 'avg_bmi', 'avg_resting_hr',  # season 제거됨
        'group_weekly_active_rate', 'user_min_steps',
        'avg_growth_rate_members_steps',  # 개인 추세 Feature 추가
        'var_growth_rate_members_steps',
        'avg_achievement_members',
        'trend_achievement_members',
        'growth_rate_steps'  # ✅ 타겟: 성장률
    ]
    training_steps = weekly_group_stats[steps_cols].dropna()

print("\n=== 샘플 데이터 (첫 10행) ===")
print(training_steps.head(10).to_string(index=False))

print("\n=== 기술 통계 ===")
print(training_steps[['group_avg_steps_14', 'next_goal_steps']].describe())

In [ ]:
# 12. 개선률 분포 시각화 (실제 데이터 패턴 확인)

import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Steps 개선률
axes[0, 0].hist(weekly_group_stats['improvement_rate_steps'], bins=50, edgecolor='black')
axes[0, 0].axvline(1.0, color='red', linestyle='--', label='유지 (1.0)')
axes[0, 0].set_title('Steps 주간 개선률 분포 (실제 데이터)')
axes[0, 0].set_xlabel('개선률')
axes[0, 0].set_ylabel('빈도')
axes[0, 0].legend()

# Kcal 개선률
axes[0, 1].hist(weekly_group_stats['improvement_rate_kcal'], bins=50, edgecolor='black', color='orange')
axes[0, 1].axvline(1.0, color='red', linestyle='--', label='유지 (1.0)')
axes[0, 1].set_title('Kcal 주간 개선률 분포')
axes[0, 1].set_xlabel('개선률')
axes[0, 1].set_ylabel('빈도')
axes[0, 1].legend()

# Duration 개선률
axes[1, 0].hist(weekly_group_stats['improvement_rate_duration'], bins=50, edgecolor='black', color='green')
axes[1, 0].axvline(1.0, color='red', linestyle='--', label='유지 (1.0)')
axes[1, 0].set_title('Duration 주간 개선률 분포')
axes[1, 0].set_xlabel('개선률')
axes[1, 0].set_ylabel('빈도')
axes[1, 0].legend()

# Distance 개선률
axes[1, 1].hist(weekly_group_stats['improvement_rate_distance'], bins=50, edgecolor='black', color='purple')
axes[1, 1].axvline(1.0, color='red', linestyle='--', label='유지 (1.0)')
axes[1, 1].set_title('Distance 주간 개선률 분포')
axes[1, 1].set_xlabel('개선률')
axes[1, 1].set_ylabel('빈도')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("\n💡 이 분포를 모델이 학습합니다!")
print("   공식이 아닌 실제 사람들의 개선 패턴입니다.")

In [ ]:
# ============================================================
# Step 7 & 9: Feature 목록 업데이트 및 학습 데이터 저장
# ============================================================
print("="*60)
print("💾 Step 7 & 9: 학습 데이터 생성 및 저장 중...")
print("="*60)

import os

# 디렉토리 생성 (없으면)
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

# ✅ Steps 학습 데이터 (13개: 입력 12개 + 타겟 1개)
steps_cols = [
    # 그룹 기본 정보 (5개)
    'member_count', 
    'group_avg_steps_14', 
    'avg_age', 
    'avg_bmi', 
    'avg_resting_hr',
    
    # 활동률 (1개)
    'group_weekly_active_rate',
    
    # 성과 피드백 (1개)
    'last_week_achievement_rate',
    
    # 개인 추세 (4개) ⭐ 핵심 추가 Feature
    'avg_growth_rate_members_steps',
    'var_growth_rate_members_steps',
    'avg_achievement_members',
    'trend_achievement_members',
    
    # 사용자 최소 기준 (1개)
    'user_min_steps',
    
    # 타겟 (1개) - 성장률
    'growth_rate_steps'
]

training_steps = weekly_group_stats[steps_cols].dropna()
training_steps.to_csv(f'{output_dir}/training_steps.csv', index=False)

print(f"\n   ✅ Steps 학습 데이터 생성 완료")
print(f"   - 총 Feature 수: {len(steps_cols) - 1}개 (입력) + 1개 (타겟)")
print(f"   - 데이터 행 수: {len(training_steps)}")
print(f"   - 파일: training_steps.csv ({len(training_steps)} rows, {len(training_steps.columns)} cols)")

# Feature 목록 출력
print("\n   📊 입력 Feature (12개):")
for i, col in enumerate(steps_cols[:-1], 1):
    print(f"      {i:2}. {col}")
print(f"\n   🎯 타겟 Feature (1개):")
print(f"       {steps_cols[-1]}")

# ✅ Kcal 학습 데이터
kcal_cols = [
    'member_count', 
    'group_avg_kcal_14', 
    'avg_age', 
    'avg_bmi', 
    'avg_resting_hr',
    'group_weekly_active_rate',
    'last_week_achievement_rate',
    'avg_growth_rate_members_kcal',
    'var_growth_rate_members_kcal',
    'avg_achievement_members',
    'trend_achievement_members',
    'user_min_kcal',
    'growth_rate_kcal'
]

training_kcal = weekly_group_stats[kcal_cols].dropna()
training_kcal.to_csv(f'{output_dir}/training_kcal.csv', index=False)
print(f"\n   ✅ training_kcal.csv: {len(training_kcal)} rows, {len(training_kcal.columns)} cols")

# ✅ Duration 학습 데이터
duration_cols = [
    'member_count', 
    'group_avg_duration_14', 
    'avg_age', 
    'avg_bmi', 
    'avg_resting_hr',
    'group_weekly_active_rate',
    'last_week_achievement_rate',
    'avg_growth_rate_members_duration',
    'var_growth_rate_members_duration',
    'avg_achievement_members',
    'trend_achievement_members',
    'user_min_duration',
    'growth_rate_duration'
]

training_duration = weekly_group_stats[duration_cols].dropna()
training_duration.to_csv(f'{output_dir}/training_duration.csv', index=False)
print(f"   ✅ training_duration.csv: {len(training_duration)} rows, {len(training_duration.columns)} cols")

# ✅ Distance 학습 데이터
distance_cols = [
    'member_count', 
    'group_avg_distance_14', 
    'avg_age', 
    'avg_bmi', 
    'avg_resting_hr',
    'group_weekly_active_rate',
    'last_week_achievement_rate',
    'avg_growth_rate_members_distance',
    'var_growth_rate_members_distance',
    'avg_achievement_members',
    'trend_achievement_members',
    'user_min_distance',
    'growth_rate_distance'
]

training_distance = weekly_group_stats[distance_cols].dropna()
training_distance.to_csv(f'{output_dir}/training_distance.csv', index=False)
print(f"   ✅ training_distance.csv: {len(training_distance)} rows, {len(training_distance.columns)} cols")

print("\n🎉 모든 학습 데이터 생성 완료!")
print(f"   저장 경로: {output_dir}/")
print(f"   생성된 파일: training_steps.csv, training_kcal.csv, training_duration.csv, training_distance.csv")
print(f"\n✨ 변경사항:")
print(f"   - ❌ season 제거됨")
print(f"   - ❌ next_goal_* 제거됨 (절대값)")
print(f"   - ✅ growth_rate_* 추가됨 (성장률)")
print(f"   - ✅ 개인 추세 Feature 4개 추가됨")

In [ ]:
# ============================================================
# Step 10: 데이터 검증 및 시각화
# ============================================================
print("\n📊 Step 10: 데이터 검증 중...")

import matplotlib.pyplot as plt
import seaborn as sns

# 10-1. Feature 분포 확인
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 개인별 성장률 평균 분포
axes[0, 0].hist(training_steps['avg_growth_rate_members_steps'], bins=50, edgecolor='black')
axes[0, 0].set_title('Distribution of avg_growth_rate_members_steps')
axes[0, 0].set_xlabel('Growth Rate')
axes[0, 0].axvline(1.0, color='red', linestyle='--', label='No Growth')
axes[0, 0].legend()

# 개인별 성장률 분산 분포
axes[0, 1].hist(training_steps['var_growth_rate_members_steps'], bins=50, edgecolor='black')
axes[0, 1].set_title('Distribution of var_growth_rate_members_steps')
axes[0, 1].set_xlabel('Variance')

# 개인별 달성률 평균 분포
axes[1, 0].hist(training_steps['avg_achievement_members'], bins=50, edgecolor='black')
axes[1, 0].set_title('Distribution of avg_achievement_members')
axes[1, 0].set_xlabel('Achievement Rate')
axes[1, 0].axvline(1.0, color='red', linestyle='--', label='100% Achievement')
axes[1, 0].legend()

# 개인별 달성률 추세 분포
axes[1, 1].hist(training_steps['trend_achievement_members'], bins=50, edgecolor='black')
axes[1, 1].set_title('Distribution of trend_achievement_members')
axes[1, 1].set_xlabel('Trend (Slope)')
axes[1, 1].axvline(0.0, color='red', linestyle='--', label='No Trend')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('../data/processed/feature_distributions.png', dpi=150)
plt.show()

print("   ✅ Feature 분포 시각화 완료 (../data/processed/feature_distributions.png)")

In [ ]:
# 10-2. 타겟 분포 확인
plt.figure(figsize=(10, 5))
plt.hist(training_steps['growth_rate_steps'], bins=50, edgecolor='black')
plt.title('Distribution of Target: growth_rate_steps')
plt.xlabel('Growth Rate')
plt.axvline(1.0, color='red', linestyle='--', label='No Growth (1.0)')
plt.legend()
plt.savefig('../data/processed/target_distribution.png', dpi=150)
plt.show()

print("   ✅ 타겟 분포 시각화 완료 (../data/processed/target_distribution.png)")

In [ ]:
# 10-3. 상관관계 확인
corr_cols = [
    'avg_growth_rate_members_steps',
    'var_growth_rate_members_steps',
    'avg_achievement_members',
    'trend_achievement_members',
    'growth_rate_steps'  # 타겟
]

correlation_matrix = training_steps[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix: Individual Trends vs Target')
plt.tight_layout()
plt.savefig('../data/processed/correlation_matrix.png', dpi=150)
plt.show()

print("   ✅ 상관관계 분석 완료 (../data/processed/correlation_matrix.png)")

In [ ]:
# 10-4. 통계 요약
print("\n📋 최종 Feature 통계 요약:")
print(training_steps[corr_cols].describe())

print("\n✅ 데이터 검증 완료!")
print(f"\n🎯 최종 데이터셋 정보:")
print(f"   - 학습 데이터 행 수: {len(training_steps)}")
print(f"   - 입력 Feature: 12개")
print(f"   - 타겟: growth_rate_steps (성장률)")
print(f"\n💡 핵심 변경사항:")
print(f"   1. 개인별 추세 Feature 4개 추가")
print(f"   2. season 제거")
print(f"   3. 절대값 → 성장률 예측으로 전환")
print(f"\n✨ 다음 단계: notebooks/03_model_training.ipynb에서 모델 학습!")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# 원본 데이터 로드
df = pd.read_csv('../data/raw/health_fitness_dataset.csv')

print(f"원본 데이터 크기: {df.shape}")
df.head()